# Mental Health AI Chatbot - ASR Streaming Server (Google Colab)

This notebook runs the **Faster Whisper ASR Streaming Server** on Google Colab with GPU acceleration.

### Setup Steps:
1. **Runtime → Change runtime type → GPU (T4)**
2. Run **Cell 1** to install dependencies
3. Upload `server_streaming_optimized.py` and both dictionary `.txt` files (Cell 2)
4. Run **Cell 3** to configure for GPU
5. Enter your **ngrok auth token** in Cell 4 (get from https://dashboard.ngrok.com/get-started/your-authtoken)
6. Run **Cell 5** to start the server
7. Run **Cell 6** to get the public WebSocket URL
8. Copy the `wss://...ngrok.../ws/asr` URL into the React app's Voice Page settings
9. Run **Cell 7** to keep the server alive

In [ ]:
# ============================================
# CELL 1: Install Dependencies
# ============================================
!pip install -q fastapi uvicorn[standard] websockets python-multipart
!pip install -q faster-whisper ctranslate2 silero-vad
!pip install -q symspellpy python-Levenshtein pyngrok nest-asyncio
print("✅ All dependencies installed!")

## Upload Files

Upload the following 3 files from the `asr/` folder in your project:
- `server_streaming_optimized.py`
- `frequency_dictionary_en_82_765.txt`
- `frequency_bigramdictionary_en_243_342.txt`

Use the file browser on the left sidebar, or run the cell below to upload.

In [ ]:
# ============================================
# CELL 2: Upload Files (if not already uploaded)
# ============================================
import os

required_files = [
    'server_streaming_optimized.py',
    'frequency_dictionary_en_82_765.txt',
    'frequency_bigramdictionary_en_243_342.txt'
]

missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f"⚠️  Missing files: {missing}")
    print("📁 Please upload them using the file browser (left sidebar) or run:")
    from google.colab import files
    uploaded = files.upload()
    print(f"✅ Uploaded: {list(uploaded.keys())}")
else:
    print("✅ All required files found!")

# Verify
for f in required_files:
    status = "✅" if os.path.exists(f) else "❌"
    print(f"  {status} {f}")

In [ ]:
# ============================================
# CELL 3: Configure for GPU (Colab)
# ============================================
import torch

with open('server_streaming_optimized.py', 'r') as f:
    content = f.read()

# Enable GPU + large-v3 model for best accuracy
content = content.replace('DEVICE = "cpu"', 'DEVICE = "cuda" if torch.cuda.is_available() else "cpu"')
content = content.replace('COMPUTE_TYPE = "int8"', 'COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"')
content = content.replace('WHISPER_MODEL = "medium"', 'WHISPER_MODEL = "large-v3"')

with open('server_streaming_optimized.py', 'w') as f:
    f.write(content)

print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"   Config: DEVICE=cuda, COMPUTE_TYPE=float16, MODEL=large-v3")
else:
    print("   ⚠️  No GPU detected! Using CPU with medium model.")
    print("   💡 Go to Runtime → Change runtime type → GPU")

In [ ]:
# ============================================
# CELL 4: Setup ngrok
# ============================================
from pyngrok import ngrok

# ⚠️ GET YOUR FREE TOKEN FROM: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # ← REPLACE THIS!

if NGROK_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE":
    print("❌ Please replace NGROK_TOKEN with your actual token!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok authenticated!")

In [ ]:
# ============================================
# CELL 5: Start ASR Server
# ============================================
import nest_asyncio
import subprocess
import threading
import time

nest_asyncio.apply()

def run_server():
    subprocess.run(['python', 'server_streaming_optimized.py'])

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("⏳ Starting ASR server (downloading & loading Whisper model...)")
print("   This may take 1-3 minutes on first run.")
time.sleep(30)
print("✅ Server is running!")

In [ ]:
# ============================================
# CELL 6: Get Public WebSocket URL
# ============================================
public_url = ngrok.connect(8000, bind_tls=True)
ws_url = str(public_url).replace('https://', 'wss://')

print()
print("=" * 70)
print("🎉 ASR SERVER IS LIVE!")
print("=" * 70)
print(f"\n📍 WebSocket URL (copy this):")
print(f"   {ws_url}/ws/asr")
print(f"\n📍 HTTP Health Check:")
print(f"   {public_url}/health")
print("\n" + "=" * 70)
print("\n📋 NEXT STEPS:")
print("   1. Copy the WebSocket URL above")
print("   2. Open your React app's Voice Page")
print("   3. Paste the WSS URL in the server URL input field")
print("   4. Click 'Connect Server' and start speaking!")
print("\n⏱️  Session expires in ~12 hours (free ngrok)")
print("=" * 70)

In [ ]:
# ============================================
# CELL 7: Keep Server Running
# ============================================
import time

print("🔄 Server is running...")
print("💡 Keep this cell running to keep the server alive")
print("⏹️  Click the stop button to shutdown\n")

counter = 0
try:
    while True:
        time.sleep(60)
        counter += 1
        print(f"⏰ Running for {counter} minutes...", end="\r")
except KeyboardInterrupt:
    print("\n✅ Server stopped!")